# Day 4 — Web Scraping + Advanced Retrieval

**Goal:** extend the corpus with scraped data, upgrade retrieval to hybrid +
reranking, and introduce evaluation metrics beyond recall@k.

Morning: scrape responsibly. Afternoon: hybrid search (BM25 + dense) with
reciprocal rank fusion and cross-encoder reranking. End of day: Ragas for
generation quality metrics.

## 1. Setup

Day 4 expands the corpus and strengthens retrieval. We now care not just about getting documents in, but about ranking evidence more effectively.

**Two themes**
- Responsible scraping and data collection.
- Stronger retrieval pipelines using hybrid search, fusion, and reranking.


In [12]:
# Load network, parsing, and retrieval dependencies in one place.
# Day 4 broadens the pipeline from local files to live web content, so setup spans both ingestion and ranking.
from dotenv import load_dotenv
load_dotenv()

import os
import time
import urllib.parse
import urllib.robotparser
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import trafilatura
import numpy as np

from utils import (
    Chunk, make_chunk_id, add_chunks,
    get_chroma_client, get_llm_client_grok,
)
from sentence_transformers import SentenceTransformer

chroma = get_chroma_client("./chroma_db")
client, generate = get_llm_client_grok()
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7594.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 2. Scraping ethics — robots.txt and rate limits

**Before writing a single line of scraper code, ALWAYS check robots.txt.**
A consultant whose scraper gets their client sued or IP-banned is a
consultant who doesn't come back to the engagement.

Scraping ethics is part of engineering quality, not a side concern.

**Why we check robots.txt and rate limits**
- Respecting site rules reduces legal and operational risk.
- Polite crawling avoids harming the source site and makes your data collection more sustainable.


In [13]:
# Check robots.txt before scraping so data collection stays respectful and compliant.
# Treat this as part of production engineering, not optional notebook etiquette.
def can_fetch(url: str, user_agent: str = "AcmeConsultingBot/1.0") -> bool:
    """Check robots.txt for permission to fetch a URL."""
    parsed = urllib.parse.urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
    except Exception as e:
        print(f"Warning: could not read {robots_url}: {e}")
        return False
    return rp.can_fetch(user_agent, url)


# Our scraping target: Python.org's events page.
# python.org has a permissive robots.txt and publishes a public events feed.
TARGET = "https://www.python.org/events/python-events/"
print(f"Can we fetch {TARGET}? → {can_fetch(TARGET)}")


Can we fetch https://www.python.org/events/python-events/? → False


### Rate limiter

Be a polite client. 1 request per 2 seconds is a reasonable default for
unknown targets. For APIs with documented limits, use those instead.

Rate limiting keeps our scraper predictable and polite.

**Concept**
- Even if a site technically allows access, flooding it with requests is still poor practice.
- A simple limiter is often enough for notebook-scale scraping.


In [14]:
# A lightweight rate limiter is enough for notebook-scale scraping.
# The goal is predictable request spacing rather than maximum throughput.
class RateLimiter:
    def __init__(self, min_interval_sec: float = 2.0):
        self.min_interval = min_interval_sec
        self.last_call = 0.0

    def wait(self):
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_call = time.time()


limiter = RateLimiter(2.0)


HEADERS = {
    # Identify yourself. Some sites block unidentified bots outright.
    "User-Agent": "AcmeConsultingBot/1.0 (+mailto:consultants@example.com)"
}


## 3. Scrape the events listing

The listing page gives us document discovery. We first gather candidate events, then follow the detail links to extract fuller text.

**Pipeline pattern**
- Discover URLs
- Fetch detail pages
- Normalize content into a shared chunk format


In [15]:
# Fetch the listing page first because it serves as our discovery layer.
# We parse the HTML into structured event records before touching the detail pages.
def fetch(url: str) -> str:
    limiter.wait()
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.text


listing_html = fetch(TARGET)
soup = BeautifulSoup(listing_html, "html.parser")

# python.org marks events with <ul class="list-recent-events"> and child <li>s
events = []
for li in soup.select("ul.list-recent-events li"):
    title_tag = li.find("h3")
    if not title_tag:
        continue
    link_tag = title_tag.find("a")
    title = title_tag.get_text(strip=True)
    href = link_tag["href"] if link_tag and link_tag.has_attr("href") else None

    time_tag = li.find("time")
    date = time_tag.get("datetime") if time_tag else None

    location_tag = li.find("span", class_="event-location")
    location = location_tag.get_text(strip=True) if location_tag else None

    events.append({
        "title": title,
        "url": urllib.parse.urljoin(TARGET, href) if href else None,
        "date": date,
        "location": location,
    })

print(f"Found {len(events)} events. First 3:")
for e in events[:3]:
    print(f"  {e['date']:<30s} {e['title']}  @ {e['location']}")


Found 6 events. First 3:
  2026-04-25T00:00:00+00:00      North Bay Python 2026  @ Petaluma, California, USA
  2026-05-13T00:00:00+00:00      PyCon US 2026  @ Long Beach, CA, USA
  2026-05-27T00:00:00+00:00      PyCon Italia 2026  @ Bologna, Italy


### Fetch each event's detail page

Use trafilatura to extract clean body text (strips nav, ads, boilerplate).

Trafilatura helps strip boilerplate and keep main article content.

**Why this matters**
- Better extraction means fewer irrelevant navigation fragments entering the index, which improves both sparse and dense retrieval.


In [16]:
# Trafilatura helps extract the main article body and discard boilerplate.
# We cap the number of detail pages here to keep the notebook quick to run.
def fetch_event_body(url: str) -> str:
    raw = fetch(url)
    body = trafilatura.extract(raw, include_comments=False, include_tables=False)
    return body or ""


# Limit to first 5 to keep notebook fast — in production you'd scrape all
event_chunks: list[Chunk] = []
for idx, ev in enumerate(events[:5]):
    if not ev["url"]:
        continue
    print(f"Fetching {ev['title']}...")
    body = fetch_event_body(ev["url"])
    text = f"EVENT: {ev['title']}\nDATE: {ev['date']}\nLOCATION: {ev['location']}\n\n{body[:2000]}"
    event_chunks.append(Chunk(
        id=make_chunk_id(ev["url"], 0, text),
        text=text,
        source=ev["url"],
        content_type="prose",
        document_type="event",
        extra={
            "event_title": ev["title"] or "",
            "event_date": ev["date"] or "",
            "event_location": ev["location"] or "",
            "scrape_timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        },
    ))

print(f"\nScraped {len(event_chunks)} event records.")


Fetching North Bay Python 2026...
Fetching PyCon US 2026...
Fetching PyCon Italia 2026...
Fetching GeoPython 2026...
Fetching EuroPython 2026...

Scraped 5 event records.


## 4. Build the combined corpus

Merge the PEP chunks (from Day 3) with the scraped events. A real engagement
often looks like this — internal docs + scraped intel in one store.

Once scraped content and local documents share a chunk format, they can live in the same retrieval system.

**Design benefit**
- A unified corpus lets the retriever surface whichever source actually answers the question, regardless of origin.


In [17]:
# Reuse recursive chunking so the scraped pages and local docs share similar retrieval units.
# A common chunk format makes hybrid indexing much easier to reason about.
import re


def chunk_recursive(text, source, max_len=1500):
    def _split(t, seps):
        if len(t) <= max_len:
            return [t]
        for sep in seps:
            if sep == "":
                return [t[i:i + max_len] for i in range(0, len(t), max_len)]
            if sep in t:
                parts = t.split(sep)
                out, buf = [], ""
                for p in parts:
                    cand = (buf + sep + p) if buf else p
                    if len(cand) <= max_len:
                        buf = cand
                    else:
                        if buf:
                            out.append(buf)
                        if len(p) > max_len:
                            out.extend(_split(p, seps[1:]))
                            buf = ""
                        else:
                            buf = p
                if buf:
                    out.append(buf)
                return out
        return [t]

    pieces = _split(text, ["\n\n", "\n", ". ", " ", ""])
    return [
        Chunk(id=make_chunk_id(source, i, p), text=p, source=source,
              content_type="prose", document_type="pep",
              extra={"chunk_idx": i})
        for i, p in enumerate(pieces)
    ]


peps = []
for p in sorted(Path("./datasets/day1").glob("*.md")):
    pep_chunks = chunk_recursive(p.read_text(), p.name)
    peps.extend(pep_chunks)

corpus = peps + event_chunks
print(f"Combined corpus: {len(peps)} PEP chunks + {len(event_chunks)} event chunks = {len(corpus)} total")


try:
    chroma.delete_collection("day4_corpus")
except Exception:
    pass

coll = chroma.create_collection(name="day4_corpus")
embs = st_model.encode([c.text for c in corpus], show_progress_bar=False)
add_chunks(coll, corpus, embeddings=embs.tolist())
print(f"Indexed in Chroma: {coll.count()} chunks")


Combined corpus: 109 PEP chunks + 5 event chunks = 114 total
Indexed in Chroma: 114 chunks


## 5. BM25 sparse retrieval

Dense embeddings are good for semantic similarity but weak on rare terms
(names, IDs, codes). BM25 is a bag-of-words ranker that's great for exact
keyword matches. Use both.

BM25 is a sparse lexical retriever. It rewards exact term overlap and often catches cases dense search misses.

**Mental model**
- Dense retrieval is good at semantic similarity.
- BM25 is good at literal matching.
- Using both usually improves robustness.


In [18]:
# BM25 provides lexical matching, which often complements dense retrieval on exact phrasing.
# Simple tokenization is enough for a teaching notebook and keeps the scoring logic transparent.
from rank_bm25 import BM25Okapi

# Tokenize — simple whitespace split is fine for BM25
def tokenize(text):
    return re.findall(r"\w+", text.lower())


tokenized_corpus = [tokenize(c.text) for c in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built.")


def bm25_search(query: str, k: int = 10) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_idx]


def dense_search(query: str, k: int = 10) -> list[tuple[int, float]]:
    q_vec = st_model.encode([query])[0].tolist()
    res = coll.query(query_embeddings=[q_vec], n_results=k)
    # Map back to corpus index by id
    id_to_idx = {c.id: i for i, c in enumerate(corpus)}
    out = []
    for ret_id, dist in zip(res["ids"][0], res["distances"][0]):
        idx = id_to_idx.get(ret_id)
        if idx is not None:
            # Convert distance to similarity score (Chroma returns cosine distance)
            out.append((idx, 1 - float(dist)))
    return out


# Smoke test
q = "pep 8 indentation"
print(f"\nQuery: {q!r}")
print("BM25 top 3:")
for i, s in bm25_search(q, 3):
    print(f"  [{s:.2f}] {corpus[i].source}: {corpus[i].text[:80]}...")
print("Dense top 3:")
for i, s in dense_search(q, 3):
    print(f"  [{s:.2f}] {corpus[i].source}: {corpus[i].text[:80]}...")


BM25 index built.

Query: 'pep 8 indentation'
BM25 top 3:
  [5.68] pep-0257.md: """Form a complex number.
Keyword arguments:
real -- the real part (default 0.0)...
  [5.15] pep-0008.md: Continuation lines should align wrapped elements either vertically using Python’...
  [4.36] pep-0008.md: One of Guido’s key insights is that code is read much more often than it is writ...
Dense top 3:
  [0.10] pep-0008.md: # PEP-0008

Source: https://peps.python.org/pep-0008/...
  [0.04] pep-0328.md: # PEP-0328

Source: https://peps.python.org/pep-0328/...
  [0.03] pep-0257.md: # PEP-0257

Source: https://peps.python.org/pep-0257/...


## 6. Reciprocal Rank Fusion (RRF)

How do we combine BM25 scores and dense similarity scores? Their scales are
different, so averaging is wrong. RRF sidesteps this: only use *ranks*.

For each document `d`, RRF score = Σ 1 / (k_rrf + rank_in_list)
where k_rrf ≈ 60 is a smoothing constant.

Reciprocal Rank Fusion combines ranked lists without needing score calibration between systems.

**Why RRF is popular**
- Dense scores and BM25 scores live on different scales.
- RRF sidesteps that issue by working with ranks instead of raw scores.


In [19]:
# RRF merges rank lists without assuming their raw scores are comparable.
# That makes it a practical default when combining dense and sparse retrievers.
def reciprocal_rank_fusion(
    result_lists: list[list[tuple[int, float]]],
    k_rrf: int = 60,
) -> list[tuple[int, float]]:
    scores: dict[int, float] = {}
    for result_list in result_lists:
        for rank, (idx, _) in enumerate(result_list):
            scores[idx] = scores.get(idx, 0.0) + 1 / (k_rrf + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def hybrid_search(query: str, k: int = 10) -> list[tuple[int, float]]:
    dense = dense_search(query, k=k * 2)
    sparse = bm25_search(query, k=k * 2)
    fused = reciprocal_rank_fusion([dense, sparse])
    return fused[:k]


print(f"\nHybrid top 3 for {q!r}:")
for i, s in hybrid_search(q, 3):
    print(f"  [RRF={s:.3f}] {corpus[i].source}: {corpus[i].text[:80]}...")



Hybrid top 3 for 'pep 8 indentation':
  [RRF=0.017] pep-0008.md: # PEP-0008

Source: https://peps.python.org/pep-0008/...
  [RRF=0.017] pep-0257.md: """Form a complex number.
Keyword arguments:
real -- the real part (default 0.0)...
  [RRF=0.016] pep-0328.md: # PEP-0328

Source: https://peps.python.org/pep-0328/...


## 7. Reranking with a cross-encoder

Bi-encoders (what we've been using) embed query and document *separately* —
fast but loses query-document interaction signal. Cross-encoders embed them
*together*, which is slower but much more accurate.

Strategy: use hybrid search to get top-20 candidates, then rerank those with
a cross-encoder to pick the final top-k. This is what Cohere Rerank does,
but we'll use a free local model.

Reranking spends extra compute on a small candidate set to improve final ordering.

**Tradeoff**
- Cross-encoders are slower than embedding lookup, but they often produce a meaningful precision jump when applied after fast retrieval.


In [22]:
# Cross-encoder reranking spends more compute on a short candidate list to improve final precision.
# It is slower than first-stage retrieval, so we only apply it after cheaper retrieval methods narrow the field.
from sentence_transformers import CrossEncoder

# bge-reranker-base is ~280MB, first load takes 30s
reranker = CrossEncoder("BAAI/bge-reranker-base")


def rerank(query: str, candidate_indices: list[int], top_k: int = 5) -> list[tuple[int, float]]:
    pairs = [[query, corpus[i].text] for i in candidate_indices]
    scores = reranker.predict(pairs)
    scored = list(zip(candidate_indices, scores.tolist()))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]


def retrieve_v2(query: str, final_k: int = 5, candidates: int = 20) -> list[dict]:
    hybrid_results = hybrid_search(query, k=candidates)
    candidate_idx = [i for i, _ in hybrid_results]
    reranked = rerank(query, candidate_idx, top_k=final_k)
    return [
        {
            "rank": r + 1,
            "score": score,
            "chunk": corpus[i],
        }
        for r, (i, score) in enumerate(reranked)
    ]


print(f"\nFull pipeline (hybrid → rerank) for {q!r}:")
for h in retrieve_v2(q):
    print(f"  rank {h['rank']} [score={h['score']:.2f}] {h['chunk'].source}")
    print(f"    {h['chunk'].text[:100]}...")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9034.12it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Full pipeline (hybrid → rerank) for 'pep 8 indentation':
  rank 1 [score=0.34] pep-0257.md
    while trimmed and not trimmed[-1]:
trimmed.pop()
while trimmed and not trimmed[0]:
trimmed.pop(0)
# ...
  rank 2 [score=0.32] pep-0008.md
    plus an opening parenthesis creates a natural 4-space indent for the
subsequent lines of the multili...
  rank 3 [score=0.11] pep-0008.md
    PEP 8 – Style Guide for Python Code
- Author:
- Guido van Rossum <guido at python.org>, Barry Warsaw...
  rank 4 [score=0.04] pep-0008.md
    One of Guido’s key insights is that code is read much more often than it is written. The guidelines ...
  rank 5 [score=0.01] pep-0008.md
    # Correct: if not seq: if seq:
# Wrong: if len(seq): if not len(seq):
- Don’t write string literals ...


## 8. Measure: does hybrid + rerank actually beat dense-only?

We'll use the Day 3 PEP eval set to measure. We expect ~10–25% recall@5 lift
from hybrid + rerank, especially on queries with rare terms.

Evaluation here asks whether retrieval sophistication actually pays off.

**What to compare**
- Dense only
- Hybrid fusion
- Hybrid plus reranking
- The goal is to earn complexity with evidence.


In [23]:
# Reuse the same eval questions so we can compare retrieval upgrades against a familiar baseline.
# Consistent evaluation makes improvements easier to trust.
eval_set = [
    {"q": "What indentation does PEP 8 recommend?",                              "source": "pep-0008.md"},
    {"q": "What is the maximum line length per PEP 8?",                          "source": "pep-0008.md"},
    {"q": "What does the Zen of Python say about explicit vs implicit?",         "source": "pep-0020.md"},
    {"q": "How should one-line docstrings be formatted?",                        "source": "pep-0257.md"},
    {"q": "What are the rules for multi-line docstrings?",                       "source": "pep-0257.md"},
    {"q": "What is an absolute import?",                                          "source": "pep-0328.md"},
    {"q": "Why were relative imports deprecated at one point?",                  "source": "pep-0328.md"},
    {"q": "What are type hints?",                                                 "source": "pep-0484.md"},
    {"q": "How does PEP 484 handle generic types?",                              "source": "pep-0484.md"},
    {"q": "How should imports be grouped?",                                       "source": "pep-0008.md"},
]


def recall_for_retriever(retriever_fn, name: str, k: int = 5) -> float:
    hits = 0
    for item in eval_set:
        results = retriever_fn(item["q"], k)
        sources = [corpus[i].source for i, _ in results]
        if item["source"] in sources:
            hits += 1
    r = hits / len(eval_set)
    print(f"  {name:25s} recall@{k} = {r:.2f}")
    return r


print("Retriever comparison:")
recall_for_retriever(dense_search, "Dense only")
recall_for_retriever(bm25_search, "BM25 only")
recall_for_retriever(hybrid_search, "Hybrid (RRF)")


def retrieve_v2_tuples(q, k):
    return [(h["chunk"].id, h["score"]) for h in retrieve_v2(q, final_k=k)]


# For rerank we need idx back — adapt
def rerank_recall(k=5):
    hits = 0
    for item in eval_set:
        results = retrieve_v2(item["q"], final_k=k, candidates=20)
        sources = [h["chunk"].source for h in results]
        if item["source"] in sources:
            hits += 1
    r = hits / len(eval_set)
    print(f"  {'Hybrid + Rerank':25s} recall@{k} = {r:.2f}")
    return r


rerank_recall()


Retriever comparison:
  Dense only                recall@5 = 1.00
  BM25 only                 recall@5 = 1.00
  Hybrid (RRF)              recall@5 = 1.00
  Hybrid + Rerank           recall@5 = 1.00


1.0

## 9. Generation quality — Ragas metrics

Recall@k tells us whether we retrieved the right *document*. It doesn't tell
us whether the *generated answer* is any good. For that we need:

- **Faithfulness**: does the answer stick to the retrieved context? (No hallucination.)
- **Answer relevance**: does the answer actually address the question?
- **Context precision**: were the retrieved chunks actually useful?

Ragas provides these out of the box. It calls an LLM as judge — so it
costs money and has its own biases — but it's a starting point.

Generation quality metrics help evaluate more than retrieval hit rate.

**Caution**
- Automated metrics can be useful, but they are still proxies. Pair them with manual spot checks, especially when answers are short or nuanced.


In [24]:
# This helper packages retrieval plus grounded generation into one callable for later evaluation.
# Keeping source snippets alongside the answer makes faithfulness inspection easier.
def answer_with_sources(question: str) -> dict:
    hits = retrieve_v2(question, final_k=4)
    contexts = [h["chunk"].text for h in hits]
    sources_text = "\n\n".join(f"[Source {i+1}]\n{c}" for i, c in enumerate(contexts))
    prompt = (
        f"Sources:\n{sources_text}\n\n"
        f"Question: {question}\n\n"
        "Answer using only the sources above. Be concise (3–5 sentences)."
    )
    ans = generate(
        system="You are a precise assistant. Ground all answers in the provided sources.",
        user=prompt,
    )
    return {"question": question, "answer": ans, "contexts": contexts}


# Build a small eval dataframe
import json

eval_samples = []
for item in eval_set[:5]:  # small subset — Ragas is expensive
    r = answer_with_sources(item["q"])
    eval_samples.append({
        "user_input": r["question"],
        "response": r["answer"],
        "retrieved_contexts": r["contexts"],
    })

print(f"Generated {len(eval_samples)} samples for Ragas eval")
print(f"Sample:\n  Q: {eval_samples[0]['user_input']}")
print(f"  A: {eval_samples[0]['response'][:200]}...")


Generated 5 samples for Ragas eval
Sample:
  Q: What indentation does PEP 8 recommend?
  A: According to PEP 8, spaces are the preferred indentation method. Python disallows mixing tabs and spaces for indentation. Spaces can be used to remain consistent with existing code that is already ind...


### Running Ragas

Ragas >= 0.2 uses the `EvaluationDataset` + `evaluate()` API. The snippet
below is the canonical shape; adjust model names to what your account has.

Ragas is optional here because it adds setup cost and API usage.

**Why include it anyway**
- It introduces a repeatable way to track faithfulness and relevance as your pipeline evolves.


In [25]:
# The Ragas section is left commented because it can be expensive and environment-dependent.
# It is included as a template for turning manual quality checks into repeatable metrics.
# Run only if you're ready for a few API calls per metric per question.
# Uncomment when ready.

from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic  # or langchain_openai

judge = LangchainLLMWrapper(ChatAnthropic(model="claude-haiku-4-5-20251001"))
dataset = EvaluationDataset.from_list(eval_samples)
result = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), ResponseRelevancy()],
    llm=judge,
)
print(result)

print("See the commented block above — uncomment when your Ragas install is ready.")


/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_16296/315139223.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_16296/315139223.py:7: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy


ModuleNotFoundError: No module named 'langchain_anthropic'

## 10. Reflection

**By now your pipeline has real retrieval quality.** Things to check:

1. Did hybrid beat dense on *every* query or just some? Look at the ones
   where dense-only did better — what was different about them?
2. Reranking adds ~100ms of latency per query. Is it worth it for your
   client's use case?
3. Ragas faithfulness below 0.8 means Claude is hallucinating. Common causes:
   generation prompt too permissive, context too fragmented, question
   ambiguous. Diagnose before you tune.
4. Your scraping pipeline works on one source. A real engagement has 5–10.
   How do you generalize without writing bespoke parsers for each?

Tomorrow: real client data, full pipeline, production checklist, demos.

Reflection should tie together data acquisition, retrieval ranking, and evaluation.

**Questions worth answering**
- Where did hybrid retrieval help most?
- When did reranking add value versus just more latency?
- What scraping choices improved or hurt corpus quality?
